# Step 4 — OpenRouter runner + smoke test

**What the runner guarantees**
- **Pinned endpoint**: each model is sent to exactly one OpenRouter endpoint (`provider.only`, `allow_fallbacks: false`), so quantization can't change mid-run.
- **Retries**: 429 / 5xx / timeouts / provider errors use exponential backoff. 401 / 402 / 403 stop the run at once (bad key, out of credits).
- **Cache**: `results/cache/<model>.jsonl`, keyed by (model, provider, exact messages, params). A rerun never pays twice.
- **Resume**: append-only `results/<dataset>/<model>__<method>.jsonl`. Already-done rows are skipped, and a half-written line from a crash is ignored.
- **Budget**: a hard USD cap on new spending per run.
- **Logged per row**: provider used, tokens in/out/reasoning, cost, latency, prompt version, params, raw output.

**Before the paid cell:** add `OPENROUTER_API_KEY` in Colab **Secrets** (🔑 icon in the left sidebar) and turn on notebook access.
Runtime: CPU.

In [1]:
# --- Setup: mount Drive, go to the project, install, import ---------------------------
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/LLM_as_Judge_Benchmark'
REPO_URL = 'https://github.com/moghbaie/LLM_as_Judge_Benchmark.git'

import os, sys, subprocess
if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
!git pull -q || true
!pip -q install -r requirements.txt
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)

Mounted at /content/drive


## 1. Tests (offline: mocked HTTP and a fake judge, no key needed)

In [2]:
!python -m pytest -q tests/test_step4_runner.py

.......................                                                  [100%]
23 passed in 11.42s


## 2. Are the pinned endpoints live, and are prices unchanged? (public, no key)

In [3]:
from ljbench import models
from ljbench.runners.openrouter import check_pin, OpenRouterClient
display(models.roster_table())
pins = pd.DataFrame([check_pin(s) for s in models.MODELS if s.backend == 'openrouter'])
pins

,key,model,tier,status,params_B,active_B,open,backend,pin,quant,$in/M,$out/M,license
0,hhem-2.1-open,HHEM-2.1-open,specialist,core,0.11,NaN,True,hf_classifier,de9a5e6ba9,fp32,NaN,NaN,apache-2.0
1,minicheck-7b,Bespoke-MiniCheck-7B,specialist,core,7.70,NaN,True,minicheck,1ed7786bcd,bf16,NaN,NaN,CC BY-NC 4.0
2,qwen3.5-4b,Qwen3.5-4B,small_open,core,4.70,NaN,True,vllm,851bf6e806,bf16,NaN,NaN,apache-2.0
3,gemma-4-e4b,Gemma 4 E4B-it,small_open,core,8.00,4.0,True,vllm,4ecc2ba63b,bf16,NaN,NaN,apache-2.0
4,gpt-oss-20b,gpt-oss-20b,mid_open,core,20.90,3.6,True,vllm,6cee5e81ee,mxfp4 (native),NaN,NaN,apache-2.0
5,qwen3.8-27b,Qwen3.8-27B,mid_open,core,27.80,NaN,True,vllm,c3a462cc3d,fp8 (official build),NaN,NaN,apache-2.0
6,muse-glimmer-30b,Muse Glimmer 30B,mid_open,optional,29.80,NaN,True,openrouter,deepinfra/bf16,bf16,0.30,1.20,apache-2.0
7,gemma-4-31b,Gemma 4 31B-it,mid_open,optional,31.30,NaN,True,openrouter,novita/bf16,bf16,0.14,0.40,apache-2.0
8,glm-5.3,GLM-5.3,large_open,core,753.00,NaN,True,openrouter,z-ai/fp8,fp8,1.40,4.40,other
9,deepseek-v4.1-flash,DeepSeek V4.1 Flash,large_open,core,763.00,NaN,True,openrouter,deepinfra/fp8,fp8,0.14,0.42,MIT


,model,ok,problem,quantization,price_in,price_out
0,muse-glimmer-30b,True,None,bf16,0.30,1.20
1,gemma-4-31b,True,None,bf16,0.14,0.40
2,glm-5.3,True,None,fp8,1.40,4.40
3,deepseek-v4.1-flash,True,None,fp8,0.14,0.42
4,muse-spark-open,True,None,unknown,1.25,4.25
5,qwen3.8-2.4t,True,None,unknown,2.00,6.00
6,kimi-k3.1-pro,True,None,mxfp4,2.00,10.00
7,qwen3.8-max,True,None,unknown,2.00,6.00
8,claude-opus-5.5,True,None,unknown,4.00,20.00
9,claude-haiku-5.5,True,None,unknown,0.10,0.50


## 3. Key check (free: reads your key's limit and usage)

In [5]:
async with OpenRouterClient() as c:
    info = await c.key_info()
# (the key label is not printed: notebooks are pushed to a public repo)
{k: info.get(k) for k in ('limit', 'limit_remaining', 'usage')}

{'limit': 50, 'limit_remaining': 50, 'usage': 0}

## 4. Smoke test: 20 held-out JudgeBench pairs × 2 orders × each core API model  (**paid, about $1–4 in total**)
Uses `judgebench_dev_20`, the prompt-debugging set, never the eval set. Set `RUN_PAID = True` to run it.

In [17]:

# ================== SET THIS TO True TO SPEND MONEY ==================
RUN_PAID = True
# =====================================================================
SMOKE_MODELS = [m.key for m in models.enabled(backend='openrouter')]   # the 9 core API models
BUDGET = {k: 1.00 for k in SMOKE_MODELS} | {'claude-opus-5.5': 2.00}  # hard cap per model, USD

from ljbench.run_api import run_one, score_dataset, RESULTS
from ljbench.runners.judge import RunConfig
print('Results folder:', RESULTS)
print('Models:', SMOKE_MODELS)

summaries = []
if not RUN_PAID:
    print()
    print('>>> RUN_PAID is False: NOTHING WAS RUN. Set RUN_PAID = True at the top of this cell and run it again.')
else:
    for key in SMOKE_MODELS:
        try:
            summaries.append(await run_one(key, 'judgebench_dev_20', 'm1', RunConfig(budget_usd=BUDGET[key])))
        except Exception as e:                      # one broken model must not hide the others
            print(f'!!! {key} crashed: {type(e).__name__}: {e}')
            summaries.append({'model': key, 'stopped': f'CRASH {type(e).__name__}: {str(e)[:200]}'})
    cols = ['model', 'written', 'call_failures', 'parse_failures', 'new_spend_usd', 'units_per_min', 'complete', 'stopped']
    display(pd.DataFrame(summaries).reindex(columns=cols))
    for s in summaries:
        for e in (s.get('errors') or [])[:2]:
            print(f"  {s['model']} error example: {e['error'][:300]}")

Results folder: /content/drive/MyDrive/LLM_as_Judge_Benchmark/results
Models: ['glm-5.3', 'deepseek-v4.1-flash', 'muse-spark-open', 'kimi-k3.1-pro', 'qwen3.8-max', 'claude-opus-5.5', 'claude-haiku-5.5', 'gpt-6-luna', 'gemini-3.8-flash']
[glm-5.3 | m1 | judgebench_dev_20] 40 units, 0 already done, 40 to run (concurrency 8)
  25/40 done | new spend $0.2944 | failed 0 | parse failures 6
[glm-5.3 | m1 | judgebench_dev_20] wrote 40, failures 0, parse failures 11, new spend $0.4527, complete=True
[deepseek-v4.1-flash | m1 | judgebench_dev_20] 40 units, 0 already done, 40 to run (concurrency 8)
  25/40 done | new spend $0.0237 | failed 0 | parse failures 4
[deepseek-v4.1-flash | m1 | judgebench_dev_20] wrote 40, failures 0, parse failures 8, new spend $0.0394, complete=True
[muse-spark-open | m1 | judgebench_dev_20] 40 units, 0 already done, 40 to run (concurrency 8)
  25/40 done | new spend $0.0497 | failed 0 | parse failures 0
[muse-spark-open | m1 | judgebench_dev_20] wrote 40, failures 0,

,model,written,call_failures,parse_failures,new_spend_usd,units_per_min,complete,stopped
0,glm-5.3,40,0,11,0.4527,4.2,True,None
1,deepseek-v4.1-flash,40,0,8,0.0394,14.1,True,None
2,muse-spark-open,40,0,0,0.0784,20.8,True,None
3,kimi-k3.1-pro,40,0,1,0.7132,5.0,True,None
4,qwen3.8-max,40,0,10,0.6324,3.9,True,None
5,claude-opus-5.5,40,0,0,0.7146,46.8,True,None
6,claude-haiku-5.5,40,0,0,0.0276,77.8,True,None
7,gpt-6-luna,40,0,0,0.0186,18.2,True,None
8,gemini-3.8-flash,40,0,0,0.1419,18.7,True,None


In [18]:
import json
from pathlib import Path
for f in sorted(Path(RESULTS, 'judgebench_dev_20').glob('*__m1.jsonl')):
    rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
    r = rows[0]
    print(f'\n#### {f.stem} | provider used: {r["provider_used"]} | gold {r["gold"]} -> label {r["label"]} '
          f'(conf {r["confidence"]}, parse {r["parse_mode"]}) | tokens out {r["tokens_out"]} '
          f'(reasoning {r["reasoning_tokens"]}) | ${r["cost_usd"]:.5f}')
    print('RAW:', r['raw_outputs'][-1][:600])


#### claude-haiku-5.5__m1 | provider used: Anthropic | gold B -> label B (conf 97.0, parse strict) | tokens out 778 (reasoning 584) | $0.00067
RAW: {"reasoning": "Jaxon lies, so Isabella (who says Jaxon tells the truth) lies, and Grace (who says Jaxon lies) tells the truth. That chain makes Emily, Devika, and Soren truthful, so Elowen (who says Devika lies) lies, Liam (who says Elowen lies) tells the truth, and Maya (who says Liam tells the truth) tells the truth. The correct answer is yes, yes, no, which Response B gives. Response A's no, no, yes contradicts this chain.", "label": "B", "confidence": 97}

#### claude-opus-5.5__m1 | provider used: Anthropic | gold A -> label A (conf 97.0, parse extracted) | tokens out 381 (reasoning 154) | $0.01883
RAW: Response A is correct: the vineyard lies, so the movie theater, the planetarium and the library all tell the truth. That makes the shopping mall a liar, and therefore the ice skating rink and the art gallery tell the truth (yes, yes, no

## 5. Look at real outputs

In [19]:
import json
from pathlib import Path
for f in sorted(Path(RESULTS, 'judgebench_dev_20').glob('*__m1.jsonl')):
    rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
    r = rows[0]
    print(f'\n#### {f.stem} | provider used: {r["provider_used"]} | gold {r["gold"]} -> label {r["label"]} '
          f'(conf {r["confidence"]}, parse {r["parse_mode"]}) | tokens out {r["tokens_out"]} '
          f'(reasoning {r["reasoning_tokens"]}) | ${r["cost_usd"]:.5f}')
    print('RAW:', r['raw_outputs'][-1][:600])


#### claude-haiku-5.5__m1 | provider used: Anthropic | gold B -> label B (conf 97.0, parse strict) | tokens out 778 (reasoning 584) | $0.00067
RAW: {"reasoning": "Jaxon lies, so Isabella (who says Jaxon tells the truth) lies, and Grace (who says Jaxon lies) tells the truth. That chain makes Emily, Devika, and Soren truthful, so Elowen (who says Devika lies) lies, Liam (who says Elowen lies) tells the truth, and Maya (who says Liam tells the truth) tells the truth. The correct answer is yes, yes, no, which Response B gives. Response A's no, no, yes contradicts this chain.", "label": "B", "confidence": 97}

#### claude-opus-5.5__m1 | provider used: Anthropic | gold A -> label A (conf 97.0, parse extracted) | tokens out 381 (reasoning 154) | $0.01883
RAW: Response A is correct: the vineyard lies, so the movie theater, the planetarium and the library all tell the truth. That makes the shopping mall a liar, and therefore the ice skating rink and the art gallery tell the truth (yes, yes, no

In [20]:
# Score (20 pairs is too few to rank models: this checks plumbing, parse rate and cost per call)
score_dataset('judgebench_dev_20').round(3)

,model,method,rows,cost_usd,tokens_out_mean,n_pairs,acc_both_orders,acc_single_order,position_consistency,share_choosing_first,parse_fail,cost_per_1000
0,claude-haiku-5.5,m1,40,0.028,983.000,20,0.95,0.975,0.95,0.475,0.000,0.690
1,claude-opus-5.5,m1,40,0.715,492.025,20,0.95,0.975,0.95,0.525,0.000,17.864
2,deepseek-v4.1-flash,m1,40,0.039,2031.025,20,0.80,0.800,0.80,0.400,0.200,0.984
3,gemini-3.8-flash,m1,40,0.142,660.850,20,1.00,1.000,1.00,0.500,0.000,3.546
4,glm-5.3,m1,40,0.453,2302.000,20,0.65,0.725,0.65,0.350,0.275,11.317
5,gpt-6-luna,m1,40,0.019,607.625,20,0.95,0.975,0.95,0.475,0.000,0.466
6,kimi-k3.1-pro,m1,40,0.713,1522.325,20,0.90,0.950,0.90,0.450,0.025,17.830
7,muse-spark-open,m1,40,0.078,96.650,20,0.80,0.900,0.80,0.500,0.000,1.960
8,qwen3.8-max,m1,40,0.632,2258.350,20,0.70,0.750,0.70,0.375,0.250,15.810


## What to check before moving on (gate for Step 6)

| Check | Pass condition |
|---|---|
| Pins live | every row `ok=True` in cell 2 |
| Provider actually used | `provider_used` matches the pinned provider for every row |
| Parse failures | < 5% per model; otherwise look at the raw outputs and fix the prompt/parser **before** freezing |
| Output length | `tokens_out` / `reasoning_tokens` per call, which tells us whether reasoning mode must be capped (it drives cost) |
| Cost | multiply measured cost per call by 3,750 calls to get the real full-run budget |

Rerunning cell 4 costs **$0**: every response comes from the cache.